# 05. 무대 위에서 — **통합**

## 이 노트북의 위치

```
기반     : NB01 (듣다)
확장     : NB02 (시각화)
협업     : NB04 (실시간 추적)
통합 ✦   : NB05 (무대) ← 여기
```

확장과 협업의 모든 산출물을 **하나의 미니 공연으로 엮습니다**.

## 재료
- 원 연주 2곡 (Satie, Prokofiev)
- NB02 학생의 cue 매핑 (확장: 시각)
- NB04 matchmaker 정렬 결과 (협업: 실시간 추적)

## 학생이 하는 것
**타임라인을 직접 편집해서** 공연의 흐름을 설계합니다.
어느 곡을 언제, 어떤 매핑으로 보여줄지 결정합니다.

`/stage` 페이지에서는 **재생 모드**(녹음 재생)와 **라이브 추적 모드**(matchmaker 연동)를 전환할 수 있습니다.

In [ ]:
from pathlib import Path
import json

ARTIFACTS = Path('artifacts')
ASSETS = Path('assets')

# 필수 산출물 확인
required = {
    'satie_audio': ASSETS / 'satie_gymnopedie_no1.wav',
    'prokofiev_audio': ASSETS / 'prokofiev_toccata_op11.wav',
    'satie_cues': ARTIFACTS / 'cues' / 'satie_visual_cues.json',
    'prokofiev_cues': ARTIFACTS / 'cues' / 'prokofiev_visual_cues.json',
}
optional = {
    'alignment_satie': ARTIFACTS / 'alignment' / 'satie_alignment.json',
    'alignment_prokofiev': ARTIFACTS / 'alignment' / 'prokofiev_alignment.json',
}

print('필수 산출물:')
for k, p in required.items():
    ok = '✅' if p.exists() else '❌'
    print(f'  {ok} {k}: {p}')

print('\n선택 산출물 (NB04):')
for k, p in optional.items():
    ok = '✅' if p.exists() else '⚠️ 없음 (라이브 추적 모드 미사용)'
    print(f'  {ok} {k}: {p}')

# 오디오 파일은 반드시 있어야 함
missing_audio = [k for k in ['satie_audio', 'prokofiev_audio'] if not required[k].exists()]
if missing_audio:
    raise FileNotFoundError(f'오디오 파일 없음: {missing_audio}')

# Cue 파일이 없으면 NB02의 cue 추출을 자동 실행
missing_cues = [k for k in ['satie_cues', 'prokofiev_cues'] if not required[k].exists()]
if missing_cues:
    print(f'\n⚠️ Cue 파일 없음 → NB01+NB02 파이프라인을 자동 실행합니다...')
    print('  (Basic Pitch 채보 + cue 추출. 수 분 소요될 수 있습니다)')
    exec(open('02_visualize.ipynb').read()) if False else None  # placeholder
    raise FileNotFoundError(
        f'다음이 없습니다: {missing_cues}.\n'
        'NB02(02_visualize.ipynb)를 먼저 실행하세요. '
        'NB02는 NB01 없이도 자동으로 채보를 수행합니다.'
    )

---
## 1. 공연 타임라인 편집

아래 `timeline` 리스트를 직접 편집하세요. 각 섹션은 **source + mapping + optional text overlay**입니다.

### source 종류
- `satie`, `prokofiev`: 원 연주
- NB03 변주 MIDI (파일명으로 직접 참조)

### mapping 종류
- `서정적`, `역동적`, `미니멀`: NB02 프리셋
- `claude_satie`, `claude_prokofiev`: NB04 Mode C의 AI 제안 매핑
- `custom`: 직접 정의

In [ ]:
# ★ 여기를 편집하세요 ★
timeline = [
    {
        'section': '도입',
        'source': 'satie',
        'start': 0.0,
        'duration': 45.0,
        'mapping': '서정적',
        'text_overlay': None,
    },
    {
        'section': '전환',
        'source': 'satie',
        'start': 45.0,
        'duration': 15.0,
        'mapping': '미니멀',
        'text_overlay': '매핑 전환: 서정적 → 미니멀',
    },
    {
        'section': '폭발',
        'source': 'prokofiev',
        'start': 0.0,
        'duration': 60.0,
        'mapping': '역동적',
        'text_overlay': None,
    },
    {
        'section': '대비',
        'source': 'prokofiev',
        'start': 60.0,
        'duration': 20.0,
        'mapping': '서정적',
        'text_overlay': '매핑 전환: 역동적 → 서정적',
    },
]

total = sum(s['duration'] for s in timeline)
print(f'⏱️  총 재생 시간: {total:.1f}초 ({total/60:.1f}분)')
print()
print('섹션 구성:')
cursor = 0.0
for s in timeline:
    print(f"  {cursor:5.1f}s → {cursor + s['duration']:5.1f}s | {s['section']:6s} | {s['source']:10s} | mapping={s['mapping']:20s}" + (f" | {s['text_overlay']}" if s['text_overlay'] else ''))
    cursor += s['duration']

---
## 2. 타임라인 시각화

각 섹션의 cue 곡선을 이어 붙여 공연 전체의 흐름을 미리 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cues_cache = {
    'satie': json.loads(required['satie_cues'].read_text()),
    'prokofiev': json.loads(required['prokofiev_cues'].read_text()),
}

fps = cues_cache['satie']['fps']
channels = ['energy', 'density', 'harmonic_tension', 'register_spread']
colors = {'satie': '#4f46e5', 'prokofiev': '#dc2626'}

fig, axes = plt.subplots(len(channels), 1, figsize=(14, 8), sharex=True)
global_t = 0.0
for section in timeline:
    src = section['source']
    if src not in cues_cache:
        continue
    frames = cues_cache[src]['frames']
    start_frame = int(section['start'] * fps)
    end_frame = int((section['start'] + section['duration']) * fps)
    segment = frames[start_frame:end_frame]
    seg_t = np.arange(len(segment)) / fps + global_t
    for ax, ch in zip(axes, channels):
        vals = [f[ch] for f in segment]
        ax.fill_between(seg_t, vals, alpha=0.6, color=colors.get(src, '#888'))
    # 섹션 경계
    for ax in axes:
        ax.axvline(global_t, color='black', linestyle='--', alpha=0.3, linewidth=0.8)
    axes[0].text(global_t + 0.3, 0.95, section['section'], fontsize=10, fontweight='bold', transform=axes[0].get_xaxis_transform())
    global_t += section['duration']

for ax, ch in zip(axes, channels):
    ax.set_ylabel(ch)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.2)

axes[-1].set_xlabel('공연 시간 (초)')
axes[0].set_title('공연 타임라인 — cue 흐름')
plt.tight_layout()
plt.show()

---
## 3. 공연 JSON 내보내기

웹 앱(`pianokit_web/`)의 `/stage` 라우트에서 이 JSON을 소비합니다.

In [ ]:
performance = {
    'title': 'pianokit performance — Satie × Prokofiev',
    'total_duration': total,
    'timeline': timeline,
    'sources': {
        'satie': {'audio': str(required['satie_audio']), 'cues': str(required['satie_cues'])},
        'prokofiev': {'audio': str(required['prokofiev_audio']), 'cues': str(required['prokofiev_cues'])},
    },
    'claude_mapping_suggestions': {},
}

out_path = ARTIFACTS / 'performance_timeline.json'
out_path.write_text(json.dumps(performance, ensure_ascii=False, indent=2))
print(f'✅ 공연 JSON 저장: {out_path}')
print(f'   {len(timeline)}개 섹션, 총 {total:.1f}초')

---
## 4. 웹 앱 연동

`pianokit_web/public/performance_timeline.json`으로 복사하면 웹 앱이 소비할 수 있습니다.

In [ ]:
import shutil

web_public = Path('pianokit_web') / 'public'
if web_public.exists():
    web_data = web_public / 'stage_data'
    web_data.mkdir(exist_ok=True)
    shutil.copy(out_path, web_data / 'performance_timeline.json')
    for key in ['satie', 'prokofiev']:
        shutil.copy(required[f'{key}_audio'], web_data / f'{key}.wav')
        shutil.copy(required[f'{key}_cues'], web_data / f'{key}_cues.json')
    # NB04 alignment 데이터 (라이브 추적 모드용, 선택적)
    for key in ['satie', 'prokofiev']:
        align_file = optional.get(f'alignment_{key}')
        if align_file and align_file.exists():
            shutil.copy(align_file, web_data / f'{key}_alignment.json')
            print(f'  ✅ {key} alignment 데이터 복사됨 (라이브 추적 모드)')
    print(f'✅ 웹 앱으로 복사됨: {web_data}')
    print(f'   웹 앱에서 fetch("/stage_data/performance_timeline.json")로 접근 가능')
else:
    print(f'⚠️ {web_public} 없음. 웹 앱 submodule이 체크아웃되어 있는지 확인하세요.')
    print(f'   수동 복사: cp {out_path} pianokit_web/public/stage_data/')

---
## 공연 체크리스트

조별 발표 전 확인:

1. ✅ 타임라인의 각 섹션이 음악적 흐름을 가지는가?
2. ✅ 각 구간의 매핑 선택이 음악 성격과 맞는가? (확장의 품질)
3. ✅ 두 곡의 대비가 공연 전체의 드라마를 만드는가?
4. ✅ 라이브 추적 모드에서 matchmaker가 연주를 잘 따라가는가?

## 확장 과제

- 같은 구간에서 **다른 매핑 프리셋**을 비교하는 A/B 섹션 만들기
- `/stage` 페이지에서 라이브 추적 모드로 직접 연주하며 시연
- 조끼리 타임라인을 교환해서 다시 편집

---

## 워크숍 마무리 질문

- **확장**: 내 연주의 어떤 차원이 가장 설득력 있게 시각화되었는가?
- **협업**: matchmaker가 내 루바토를 제대로 따라갔는가? 어느 곡이 더 어려웠는가?